# Database overview notebook

> First time use: follow instructions in the README.md file in this directory.

In [2]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()


Timelink version: 1.1.26
Project name: dehergne-repertoire
Project home: /Users/jrc/develop/dehergne-repertoire
Database type: sqlite
Database name: dehergne_repertoire
Kleio image: timelinkserver/kleio-server
Kleio server token: KAOI9...
Kleio server URL: http://127.0.0.1:8088
Kleio server home: /Users/jrc/develop/dehergne-repertoire
Kleio server container: great_greider
Kleio version requested: latest
Kleio server version: 12.9.588 (2025-06-06 16:04:22)
SQLite directory: /Users/jrc/develop/dehergne-repertoire/database/sqlite
Database version: 6ccf1ef385a6
Call print_info(show_token=True) to show the Kleio Server token
Call print_info(show_password=True) to show the Postgres password
TimelinkNotebook(project_name=dehergne-repertoire, project_home=/Users/jrc/develop/dehergne-repertoire, db_type=sqlite, db_name=dehergne_repertoire, kleio_image=timelinkserver/kleio-server, kleio_version=latest, postgres_image=postgres, postgres_version=latest)


## Ligar à base de dados

---

## Connect to database

In [3]:
from timelink.api.database import TimelinkDatabase

# to connect to a postgresql database
# do: db_type='postgresql' and db_name='timelink' and db_user='postgres' and db_password='postgres' and db_host='localhost'

db: TimelinkDatabase = tlnb.db

## Database status

In [4]:
tlnb.table_row_count_df()

,table,count
0,acts,28
1,alembic_version,1
2,aregisters,0
3,attributes,26819
4,blinks,98
5,class_attributes,58
6,classes,10
7,entities,32959
8,geoentities,359
9,kleiofiles,29


## Recent sources

In [5]:
number_of_sources = 100

Source = db.get_model('source')

with db.session() as session:
    sources = session.query(Source).order_by(Source.updated.desc()).all()
    for source in sources[:number_of_sources]:
        print(f"{source.id:32} {source.updated:%Y-%m-%d %H:%M:%S} {source.kleiofile}")


dehergne-b                       2025-09-12 07:03:47 /kleio-home/sources/dehergne-b.cli
dehergne-r                       2025-09-12 07:02:13 /kleio-home/sources/dehergne-r.cli
dehergne-l                       2025-09-12 07:02:06 /kleio-home/sources/dehergne-l.cli
dehergne-a                       2025-09-12 07:02:01 /kleio-home/sources/dehergne-a.cli
dehergne-y                       2025-06-09 04:44:14 /kleio-home/sources/dehergne-y.cli
dehergne-x                       2025-06-09 04:44:12 /kleio-home/sources/dehergne-x.cli
dehergne-w                       2025-06-09 04:44:10 /kleio-home/sources/dehergne-w.cli
dehergne-v                       2025-06-09 04:44:05 /kleio-home/sources/dehergne-v.cli
dehergne-t                       2025-06-09 04:44:00 /kleio-home/sources/dehergne-t.cli
dehergne-s                       2025-06-09 04:43:53 /kleio-home/sources/dehergne-s.cli
dehergne-p                       2025-06-09 04:43:39 /kleio-home/sources/dehergne-p.cli
dehergne-o                      

### Tipos de attributos

---

### Attribute types

In [6]:
import pandas as pd

from sqlalchemy import func
from sqlalchemy import select


pd.set_option('display.max_rows', 500)

attr_table = tlnb.db.get_table('attributes')
tlnb.db.describe('attributes', show=True)
print()
stmt = select(
    attr_table.c.the_type,
    func.count().label('count'),
    func.count(func.distinct(attr_table.c.the_value)).label('distinct_value')
    ).group_by('the_type')
print(stmt)
print()

with tlnb.db.session() as session:
    # nml2 = session.query(Attribute.the_type,func.count().label('tot')).group_by(Attribute.the_type).all()
    nml = session.execute(stmt)
    attribute_df = pd.DataFrame(nml)

attribute_df

attributes (model_table)
id                   entities             VARCHAR    
class                entities             VARCHAR    
inside               entities             VARCHAR    {ForeignKey('entities.id')}
the_source           entities             VARCHAR    
the_order            entities             INTEGER    
the_level            entities             INTEGER    
the_line             entities             INTEGER    
groupname            entities             VARCHAR    
extra_info           entities             JSON       
updated              entities             DATETIME   
indexed              entities             DATETIME   
id                   attributes           VARCHAR    {ForeignKey('entities.id')}
entity               attributes           VARCHAR    {ForeignKey('entities.id')}
the_type             attributes           VARCHAR    
the_value            attributes           VARCHAR    
the_date             attributes           VARCHAR    
obs                  attribute

,the_type,count,distinct_value
0,activa,286,3
1,alternative-name,4,4
2,alternative-name@wikidata,3,3
3,baptizado,28,28
4,baptizado@wikidata,26,26
5,bibliografia,9,9
6,cargo,356,225
7,chegada,487,71
8,chegada@wikidata,422,63
9,dehergne,1454,1211


## Vocabulary of attributes

In [7]:
from timelink.pandas import attribute_values

df_totals = attribute_values('jesuita-entrada',db=tlnb.db)
df_totals

,count,date_min,date_max
value,,,
?,481,1544,17880828
Coimbra,63,15420000,17420427
Lisboa,45,1546,17530612
Paris,41,16300108,17590310
Roma,40,15480311,17560709
Goa,33,1548,1736
Évora,26,15660714,17460405
Nancy,15,16270929,17510827
Landsberg,14,16230729,17571009


## Mostrar uma fonte

---

## Show a source

In [8]:
id = 'dehergne-locations-1644'
with db.session() as session:
    source = session.get(Source,id)
    print(source.to_kleio())

fonte$dehergne-locations-1644/0/type=geoinformation/ref=None/loc=None/kleiofile="/kleio-home/sources/dehergne-locations-1644.cli""/replaces=None/obs="""
      Planche: Carte des Chrétientés Chinoises de la fin des Ming (1644).p. 353

      _Diocèse_
      Depuis 1514, l'évêché de Funchal (île de Madère)
      s'étend jusqu'aux extrémi-
      tés de l'Asie. Depuis le 3 nov. 1534 est
      créé l'évêché de Goa, dont est déta-
      ché, le 23 janv. 1576, le diocèse de
      Macao, qui sera amputé, 14 février
      1588, du Japon (évêché de Funai):
      l'évêché de Macao comprend théori-
      quement tout l'empire de Chine.

      _Provinces Jésuites_

      la vice-province du Japon (province en
      1611); en 1623, la vice-province de Chine
      indépendante du Japon en est déta-
      chée; mais un visiteur jésuite contrôle
      l'une et l'autre.

      La Chine impériale distingue dans chaque province civile les _fou_, préfec-
      tures ou villes de premier ordre, dont dépenden

## Obter modelos para acesso aos dados

---

## Obtain models for data access

In [9]:
db.get_models_ids()


['entity',
 'attribute',
 'relation',
 'act',
 'source',
 'aregister',
 'person',
 'object',
 'geoentity',
 'rentity',
 'class']

In [10]:
Model = db.get_model('geoentity')

with db.session() as session:
    entities = session.query(Model).all()
    for entity in entities[:10]:
        print(entity.to_kleio())
        print()

geo1$Chekiang#Tche-kiang, today Zhejiang, 浙江, @wikidata:Q16967 @dehergne:396/geo1
  atr$geoentity:name@wikidata/"https://www.wikidata.org/wiki/Q16967"#Tche-kiang, today Zhejiang, 浙江, @wikidata:Q16967 @dehergne:396%Q16967/1644
  atr$geoentity:name@dehergne/"https://archive.org/details/bhsi37/page/n396/mode/1up"#Tche-kiang, today Zhejiang, 浙江, @wikidata:Q16967 @dehergne:396%396/1644

geo2$Hangchou#Hang-tcheou, today Hangzhou, 杭州, @wikidata:Q4970/geo2
  atr$activa/sim/1611
  atr$residencia-missao/Jesuíta/1611
  atr$geoentity:name@wikidata/"https://www.wikidata.org/wiki/Q4970"#Hang-tcheou, today Hangzhou, 杭州, @wikidata:Q4970%Q4970/1644

geo3$Fuyang#Fou-yang, today Fuyang, 富阳, @wikidata:Q1011103/geo3
  atr$activa/sim/1642
  atr$geoentity:name@wikidata/"https://www.wikidata.org/wiki/Q1011103"#Fou-yang, today Fuyang, 富阳, @wikidata:Q1011103%Q1011103/1644

geo3$Jenho#Jen-houo, today Renhe, 仁和县 (@wikidata:Q9385136), Historical county name, coordinates: 30.448897N, 120.307504E/geo3/obs=no wikidat